# Turn AC-VFINAL-ALIGNMENT-AND-DISCOVERY, section O -- Colab CUDA GPU benchmark runner

Embeds the pinned 5,000-sample benchmark set with `nlpai-lab/KURE-v1` (revision
`4ed4540949c70b7da2c74004a915e1f2d5e46e4f`, dimension 1024, float32, L2-normalized)
and writes a JSON Lines result file matching
`domain/agent-comparison/four-arm-ac/gpu-benchmark-result.schema.json`.

**Inputs (upload to Google Drive before running):**
- `embedding-input-fulltext.jsonl` (task-owned; NOT committed to git -- see
  `scripts/p11f0-embedding-input-manifest.mjs`'s own output)
- `embedding-input-manifest.summary.json` (git-tracked; carries `benchmark_sample.sample_ids`)

**Output:** `gpu-benchmark-result.colab-cuda.jsonl`, downloaded and verified locally with
`node scripts/p11f0-gpu-benchmark-result-verify.mjs <result> <summary>` -- this Turn does
NOT execute this notebook or upload anything; it is a runnable template only.

In [ ]:
!pip install -q sentence-transformers==3.0.1
from google.colab import drive
drive.mount("/content/drive")

import json, time
from sentence_transformers import SentenceTransformer
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"

MODEL_REPOSITORY = "nlpai-lab/KURE-v1"
MODEL_REVISION = "4ed4540949c70b7da2c74004a915e1f2d5e46e4f"
DIMENSION = 1024
RUNNER = "COLAB_CUDA"
DRIVE_DIR = "/content/drive/MyDrive/p11f0-embedding-manifest"

model = SentenceTransformer(MODEL_REPOSITORY, revision=MODEL_REVISION, device="cuda")
assert model.get_sentence_embedding_dimension() == DIMENSION

In [ ]:
with open(f"{DRIVE_DIR}/embedding-input-manifest.summary.json") as f:
    summary = json.load(f)
sample_ids = set(summary["benchmark_sample"]["sample_ids"])

sample_rows = []
with open(f"{DRIVE_DIR}/embedding-input-fulltext.jsonl") as f:
    for line in f:
        row = json.loads(line)
        if row["embedding_input_id"] in sample_ids:
            sample_rows.append(row)
assert len(sample_rows) == len(sample_ids), f"expected {len(sample_ids)} sample rows, found {len(sample_rows)}"

In [ ]:
out_path = f"{DRIVE_DIR}/gpu-benchmark-result.colab-cuda.jsonl"
with open(out_path, "w") as out:
    for row in sample_rows:
        started = time.perf_counter()
        vector = model.encode(row["text"], normalize_embeddings=True).tolist()
        elapsed_ms = (time.perf_counter() - started) * 1000
        result = {
            "embedding_input_id": row["embedding_input_id"],
            "embed_text_sha256": row["embed_text_sha256"],
            "model_repository": MODEL_REPOSITORY,
            "model_revision": MODEL_REVISION,
            "dimension": DIMENSION,
            "dtype": "float32",
            "normalization": "l2",
            "vector": vector,
            "runner": RUNNER,
            "elapsed_ms": elapsed_ms,
        }
        out.write(json.dumps(result) + "\n")
print(f"wrote {len(sample_rows)} results to {out_path}")